# Geospatial ETL analyst walkthrough

This notebook reuses `examples.geospatial_etl.workflow` so the script remains the canonical implementation. Use the notebook after `python examples/geospatial_etl/run_etl.py` is already working.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
if (cwd / "examples" / "geospatial_etl" / "workflow.py").exists():
    REPO_ROOT = cwd
    EXAMPLE_DIR = cwd / "examples" / "geospatial_etl"
elif (cwd / "workflow.py").exists() and (cwd / "data" / "demo_sites.csv").exists():
    EXAMPLE_DIR = cwd
    REPO_ROOT = cwd.parents[1]
else:
    raise RuntimeError("Open this notebook from the repo root or examples/geospatial_etl/.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from honua_sdk import HonuaClient
from examples.geospatial_etl.workflow import (
    build_demo_where,
    build_upsert_plan,
    dataframe_to_source_geodataframe,
    load_source_dataframe,
    normalize_source_geodataframe,
    query_target_snapshot,
    run_workflow,
    validate_source_geodataframe,
)

BASE_URL = "http://localhost:8080"
SERVICE_ID = "test_service"
LAYER_ID = 0
INPUT_PATH = EXAMPLE_DIR / "data" / "demo_sites.csv"
OUTPUT_DIR = EXAMPLE_DIR / "output"
WHERE = build_demo_where()


## Inspect the source feed

In [ ]:
source_frame = load_source_dataframe(INPUT_PATH)
source_frame

## Transform, validate, and preview the upsert plan

In [ ]:
source_gdf = dataframe_to_source_geodataframe(source_frame)

with HonuaClient(BASE_URL) as client:
    pre_load = query_target_snapshot(
        client,
        service_id=SERVICE_ID,
        layer_id=LAYER_ID,
        where=WHERE,
    )

transformed_gdf = normalize_source_geodataframe(source_gdf, target_crs=pre_load.target_crs)
validation = validate_source_geodataframe(transformed_gdf)
plan = build_upsert_plan(validation.valid_gdf, pre_load.geodataframe, target_crs=pre_load.target_crs)

print(f"Target CRS: {pre_load.target_crs}")
print(f"Target records before load: {pre_load.feature_count}")
print(f"Valid rows: {validation.valid_count}")
print(f"Rejected rows: {validation.rejected_count}")
print(f"Planned adds: {plan.add_count}")
print(f"Planned updates: {plan.update_count}")

display(transformed_gdf[["uid", "name", "status", "count", "geometry"]])
display(pd.DataFrame([issue.to_dict() for issue in validation.rejected_rows]))


## Run the shared ETL workflow and inspect the artifacts

In [ ]:
with HonuaClient(BASE_URL) as client:
    result = run_workflow(
        client,
        base_url=BASE_URL,
        service_id=SERVICE_ID,
        layer_id=LAYER_ID,
        input_path=INPUT_PATH,
        output_dir=OUTPUT_DIR,
    )

result.summary

In [ ]:
if result.preview_path is not None:
    display(Image(filename=result.preview_path))
else:
    print("No preview artifact was produced.")
